# 01 — Verify sources, choose an orbit, export a small real pilot

Run this in **Google Colab** using your own authorized Earth Engine account. Start with CPU. The original repository project ID is supplied as a non-secret default. We use the version-005 single-product period **2017–2022** to avoid an unverified version splice. The notebook checks actual availability; it never silently falls back to a different product. GAUL district boundaries require name/licence review. No exports start until the final launch flag is changed.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, json
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path("/content/cht-soil-moisture-gapfilling")]
             if (p / "pyproject.toml").exists()), None)
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/cht-soil-moisture-gapfilling")
    subprocess.run(["git", "clone", "--branch", "research-notebooks-v2", "--single-branch",
                    "https://github.com/Bipul-roy-AI/cht-soil-moisture-gapfilling.git", str(ROOT)], check=True)
if ROOT is None:
    raise RuntimeError("Open this notebook from the cloned repository, or run it in Colab.")
ROOT = ROOT.resolve()
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[gee,notebooks]"], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT)


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/CHT_SM_v2")
else:
    WORK = ROOT / "work_v2"
WORK.mkdir(parents=True, exist_ok=True)
print("Persistent work folder:", WORK)


In [ ]:
from chtsm.data import read_config, save_json
config = read_config(ROOT / "configs/study_v2.json")
# This is the non-secret Cloud project ID used in your original notebook.
GEE_PROJECT_ID = "gee-projects-475506"  # Change if that project is no longer authorized.
import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_ID)
from chtsm.gee import inventory
info, cells = inventory(config)
print(json.dumps(info, indent=2))
grid_features = cells.getInfo()["features"]
import pandas as pd
cell_table = pd.DataFrame([f["properties"] for f in grid_features])
display(cell_table[["cell_id", "lon", "lat"]])

Choose one relative orbit from the inventory and inspect its coverage. An orbit with many scenes may still have incomplete coverage. The pilot uses the first few grid cells as a software/data-feasibility check, not a representative forest sample. Later full exports cover every selected district grid centre.

In [ ]:
RELATIVE_ORBIT = None  # Replace with one integer shown in relative_orbit_scene_counts.
if RELATIVE_ORBIT is None:
    raise ValueError("Choose an orbit from the printed inventory, set RELATIVE_ORBIT, then rerun this cell.")
config["relative_orbit"] = int(RELATIVE_ORBIT)
save_json(WORK / "study_config.json", config)
save_json(WORK / "inventory.json", info)
cell_table.to_csv(WORK / "target_grid.csv", index=False)
print("Saved the actual selected export configuration and grid.")

In [ ]:
from chtsm.gee import start_month_export
START_PILOT_EXPORT = False
if START_PILOT_EXPORT:
    pilot = start_month_export(config, "2021-06-01", "2021-06-08",
        folder="CHT_SM_v2_exports", prefix="chtsm_v2_pilot_20210601", pilot_cells=4)
    save_json(WORK / "pilot_task.json", pilot)
    print(pilot)
else:
    print("Set START_PILOT_EXPORT=True after reviewing the inventory and selected orbit.")

Monitor the task at https://code.earthengine.google.com/tasks. The CSV appears in your Google Drive folder `CHT_SM_v2_exports`. Before launching years of exports, check that the pilot contains plausible values, both raw and terrain-normalized SAR, real EVI, QA and acquisition IDs. Use notebook 03's pilot section. If it fails, share the task error and inventory/QA summary; do not send credentials. This authoring environment has not run an authenticated GEE export.